In [1]:
import subprocess
result = subprocess.run(["pip", "install", "kaggle", "-q"], capture_output=True, text=True)
print("✅ Kaggle CLI installed")

✅ Kaggle CLI installed


In [2]:
import os, json

os.makedirs("/root/.kaggle", exist_ok=True)

# Version cũ cần JSON với 2 field username + key
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({
        "username": "lam le191",              # ← điền vào
        "key": "KGAT_c59b6f5137e14aeaf1a4673f7fcf03a2" # ← token của bạn
    }, f)

os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("✅ kaggle.json created")

✅ kaggle.json created


In [3]:
import subprocess

result = subprocess.run(
    ["kaggle", "datasets", "list", "--search", "yelp-dataset"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

ref                                                       title                                                    size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------------------  ------------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
yelp-dataset/yelp-dataset                                 Yelp Dataset                                       4374983563  2022-03-17 22:59:01.257000         163039       1799  0.75             
ilhamfp31/yelp-review-dataset                             Yelp Review Sentiment Dataset                       169591198  2020-01-29 06:22:07                  4794         37  0.7058824        
omkarsabnis/yelp-reviews-dataset                          Yelp Reviews Dataset                                  3656661  2018-06-03 04:45:50                 11085         48  0.29411766       
fireballbyedimyrnmom/yelp-dataset  

In [4]:
import subprocess, os, shutil

data_dir = "/home/iceberg/data/yelp"
tmp_dir  = "/home/iceberg/data/yelp_tmp"  # giải nén ra chỗ khác

# Dọn sạch
shutil.rmtree(data_dir, ignore_errors=True)
shutil.rmtree(tmp_dir,  ignore_errors=True)
os.makedirs(data_dir)
os.makedirs(tmp_dir)

files_to_download = [
    "yelp_academic_dataset_business.json",
    "yelp_academic_dataset_user.json",
]

for filename in files_to_download:
    zip_name = filename.replace(".json", ".zip")
    zip_path = f"{data_dir}/{zip_name}"

    print(f"\n{'='*60}")
    print(f"⬇️  Downloading: {filename}")
    print(f"{'='*60}")

    # Download — KHÔNG dùng --unzip, tự giải nén sau
    process = subprocess.Popen(
        [
            "kaggle", "datasets", "download",
            "yelp-dataset/yelp-dataset",
            "--file", filename,
            "-p", data_dir,
            # KHÔNG có --unzip
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    process.wait()

    # Tìm file vừa download (có thể là .zip hoặc .json)
    downloaded = [f for f in os.listdir(data_dir)]
    print(f"Files hiện có: {downloaded}")

    # Tìm file zip
    zip_files = [f for f in os.listdir(data_dir) if f.endswith(".zip")]
    json_files = [f for f in os.listdir(data_dir) if f.endswith(".json")]

    if zip_files:
        zf = f"{data_dir}/{zip_files[0]}"
        print(f"📦 Giải nén: {zip_files[0]} → {tmp_dir}")
        result = subprocess.run(
            ["unzip", "-o", zf, "-d", tmp_dir],
            capture_output=True, text=True
        )
        print(result.stdout[:500])
        os.remove(zf)  # xóa zip

        # Move JSON từ tmp về data_dir
        for f in os.listdir(tmp_dir):
            if f.endswith(".json"):
                src = f"{tmp_dir}/{f}"
                dst = f"{data_dir}/{f}"
                shutil.move(src, dst)
                size_mb = os.path.getsize(dst) / (1024**2)
                print(f"✅ {f}: {size_mb:,.1f} MB")
    elif json_files:
        # Kaggle CLI đã tự giải nén
        for jf in json_files:
            size_mb = os.path.getsize(f"{data_dir}/{jf}") / (1024**2)
            print(f"✅ {jf}: {size_mb:,.1f} MB (đã là JSON)")

# Verify cuối
print(f"\n{'='*60}")
print("📂 Kết quả cuối:")
for f in sorted(os.listdir(data_dir)):
    size_mb = os.path.getsize(f"{data_dir}/{f}") / (1024**2)
    print(f"  {f:50s} {size_mb:,.1f} MB")

shutil.rmtree(tmp_dir, ignore_errors=True)


⬇️  Downloading: yelp_academic_dataset_business.json
Dataset URL: https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset
License(s): other

  0%|          | 0.00/20.8M [00:00<?, ?B/s]
 96%|█████████▌| 20.0M/20.8M [00:00<00:00, 203MB/s]
100%|██████████| 20.8M/20.8M [00:00<00:00, 197MB/s]

Files hiện có: ['yelp_academic_dataset_business.json']
✅ yelp_academic_dataset_business.json: 20.8 MB (đã là JSON)

⬇️  Downloading: yelp_academic_dataset_user.json
Dataset URL: https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset
License(s): other

  0%|          | 0.00/1.84G [00:00<?, ?B/s]
  1%|          | 20.0M/1.84G [00:00<00:09, 201MB/s]
  2%|▏         | 41.0M/1.84G [00:00<00:09, 212MB/s]
  3%|▎         | 62.0M/1.84G [00:00<00:08, 212MB/s]
  4%|▍         | 83.0M/1.84G [00:00<00:08, 215MB/s]
  6%|▌         | 104M/1.84G [00:00<00:08, 214MB/s] 
  7%|▋         | 125M/1.84G [00:00<00:08, 215MB/s]
  8%|▊         | 146M/1.84G [00:00<00:08, 205MB/s]
  9%|▉         | 166M/1.84G [00:00<00:09, 200M

In [5]:
import subprocess, os, shutil



# Kiểm tra magic bytes thực tế
print("🔍 Kiểm tra magic bytes:\n")
for f in sorted(os.listdir(data_dir)):
    fpath = f"{data_dir}/{f}"
    size_mb = os.path.getsize(fpath) / (1024**2)
    with open(fpath, 'rb') as fp:
        magic = fp.read(4)
    print(f"  {f}")
    print(f"  Size : {size_mb:,.1f} MB")
    print(f"  Magic: {magic} → {'ZIP 📦' if magic[:2] == b'PK' else 'JSON ✅' if magic[:1] == b'{' else 'Unknown'}")
    print()

🔍 Kiểm tra magic bytes:

  yelp_academic_dataset_business.json
  Size : 20.8 MB
  Magic: b'PK\x03\x04' → ZIP 📦

  yelp_academic_dataset_user.json
  Size : 1,885.0 MB
  Magic: b'PK\x03\x04' → ZIP 📦



In [6]:
import subprocess, os, shutil



os.makedirs(tmp_dir, exist_ok=True)

for f in sorted(os.listdir(data_dir)):
    fpath = f"{data_dir}/{f}"
    path= f"{tmp_dir}/{f}"
    with open(fpath, 'rb') as fp:
        magic = fp.read(2)
    
    if magic != b'PK':
        size_mb = os.path.getsize(fpath) / (1024**2)
        print(f"⏭️  {f} — đã là JSON thật ({size_mb:,.1f} MB), bỏ qua")
        continue

    size_mb = os.path.getsize(fpath) / (1024**2)
    print(f"📦 Giải nén: {f} ({size_mb:,.1f} MB) ...")

    # Giải nén vào tmp_dir
    result = subprocess.run(
        ["unzip", "-o", fpath, "-d", tmp_dir],
        capture_output=True, text=True
    )
    
    if result.returncode != 0:
        print(f"❌ Lỗi unzip: {result.stderr}")
        continue

    # Move tất cả JSON từ tmp về data_dir
    for extracted in os.listdir(tmp_dir):
        src = f"{tmp_dir}/{extracted}"
        dst = f"{data_dir}/{extracted}"
        shutil.move(src, dst)
        size_mb = os.path.getsize(dst) / (1024**2)
        print(f"  ✅ Extracted: {extracted} → {size_mb:,.1f} MB")

    

# Dọn tmp
shutil.rmtree(tmp_dir, ignore_errors=True)

# Verify cuối
print("📂 Kết quả cuối:\n")
for f in sorted(os.listdir(data_dir)):
    fpath = f"{data_dir}/{f}"
    size_mb = os.path.getsize(fpath) / (1024**2)
    with open(fpath, 'rb') as fp:
        magic = fp.read(1)
    status = "✅ JSON" if magic == b'{' else "⚠️  Không phải JSON"
    print(f"  {status}  {f:50s} {size_mb:,.1f} MB")

📦 Giải nén: yelp_academic_dataset_business.json (20.8 MB) ...
  ✅ Extracted: yelp_academic_dataset_business.json → 113.4 MB
📦 Giải nén: yelp_academic_dataset_user.json (1,885.0 MB) ...
  ✅ Extracted: yelp_academic_dataset_user.json → 3,207.5 MB
📂 Kết quả cuối:

  ✅ JSON  yelp_academic_dataset_business.json                113.4 MB
  ✅ JSON  yelp_academic_dataset_user.json                    3,207.5 MB
